# Practice Assignment 9 — ML Application Orchestration and Deployment

## Context

This assignment uses the Week 9 sentiment-analysis application in
`Lecture/week-9/ml-app-docker`. The application exposes a FastAPI service,
packages a Transformers model in Docker, and deploys the container to
Kubernetes.

## Learning Outcomes

After completing this notebook, you should be able to:

1. Explain the lifecycle of an ML inference API.
2. Test FastAPI health and prediction endpoints.
3. Build, inspect, and run a Docker image.
4. Explain Dockerfile instructions and container health checks.
5. Read Kubernetes Namespaces, ConfigMaps, Deployments, Services, PVs, and PVCs.
6. Deploy and inspect an ML service with `kubectl`.
7. Explain startup, readiness, and liveness probes.
8. Explain how an HPA scales a deployment.

Attempt every TODO before consulting the Week 9 lecture material.

## Setup

Run this notebook from the root of the ML application project. All files and
commands below use the notebook's current working directory (`./`).

In [ ]:
from pathlib import Path
import json
import yaml

APP_DIR = Path.cwd()
print("Application directory:", APP_DIR.resolve())
print("Files:")
for path in sorted(APP_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(APP_DIR))

# Part 1 — Understand the FastAPI Application

## Task 1: Inspect the API contract

Read `app/main.py` and identify the model name, request schema, response
schema, and the behavior of `/health` and `/predict`. Explain why model loading
is placed in the FastAPI lifespan rather than inside every request.

In [ ]:
# TODO: Read and inspect APP_DIR / "app/main.py".
# TODO: Print the model name, request fields, response fields, and endpoint behavior in your own words.

## Task 2: Run and test the service locally

Start the service with Uvicorn, then test the health endpoint and send at least
two texts to `/predict`. Record one successful response and explain what a
503 response means before the model has finished loading.

In [ ]:
# TODO: From APP_DIR, run the service with:
# uvicorn app.main:app --host 0.0.0.0 --port 8000
# TODO: In another terminal, run curl commands for /health and /predict.

# Part 2 — Containerize the ML Service

## Task 3: Explain the Dockerfile

Read the supplied `Dockerfile`. Explain the purpose of: the base image,
environment variables, `WORKDIR`, dependency installation, the non-root user,
model pre-download step, `EXPOSE`, `HEALTHCHECK`, and `CMD`.

In [ ]:
# TODO: Print the Dockerfile and annotate each important instruction.

## Task 4: Build and run the image

Build the image as `sentiment-api:week9`, run it on host port 8000, and test
both endpoints. Inspect the image size and container logs. If Docker is not
available, write the commands and explain the expected result.

In [ ]:
# TODO: Execute from APP_DIR:
# docker build -t sentiment-api:week9 .
# docker run --rm --name sentiment-api -p 8000:8000 sentiment-api:week9
# TODO: Test /health and /predict, then inspect with docker images and docker logs.

# Part 3 — Kubernetes Configuration

## Task 5: Parse and summarize manifests

Load every YAML manifest under `APP_DIR / "k8s"` and create a table containing
the filename, Kubernetes kind, object name, and namespace. Explain why the
Namespace and ConfigMap are separate objects.

In [ ]:
# TODO: Load the YAML files with yaml.safe_load and print the requested summary.

## Task 6: Analyze the Deployment

Inspect `deployment.yaml`. Identify the desired replica count, container port,
image, environment source, CPU/memory requests and limits, probes, and mounted
volume. Explain how readiness differs from liveness and startup checks.

In [ ]:
# TODO: Extract the deployment fields programmatically and print them.
# TODO: Write a short explanation of the three probes.

## Task 7: Deploy to Kubernetes

Apply the manifests in this order: namespace, ConfigMap, storage, Deployment,
Service, and HPA. Inspect pods, deployment rollout status, service details,
events, and logs. Port-forward the Service and test the API locally.

In [ ]:
# TODO: Run from APP_DIR:
# kubectl apply -f k8s/namespace.yaml
# kubectl apply -f k8s/configmap.yaml -f k8s/pv.yaml -f k8s/deployment.yaml -f k8s/service.yaml -f k8s/hpa.yaml
# kubectl -n ml-app get all,pvc
# kubectl -n ml-app rollout status deployment/ml-app
# kubectl -n ml-app port-forward service/ml-app 8000:80
# TODO: Test /health and /predict through the port-forward.

# Part 4 — Storage, Scaling, and Design Questions

## Task 8: Explain persistent storage

Describe the relationship between the PersistentVolume, PersistentVolumeClaim,
and Deployment volume mount. Discuss one limitation of the supplied `hostPath`
volume for a multi-node production cluster.

In [ ]:
# TODO: Answer the storage questions in your own words.

## Task 9: Analyze autoscaling

Inspect `hpa.yaml`. State the minimum and maximum replicas, the target metric,
and the condition that causes scaling. Explain why CPU utilization may be an
imperfect proxy for inference latency or request throughput.

In [ ]:
# TODO: Parse hpa.yaml and print its scaling configuration.
# TODO: Answer the design question.

## Task 10: Troubleshooting and cleanup

For each symptom, give one diagnostic command and one likely cause:

1. The pod stays in `Running` but is not added to the Service endpoints.
2. The pod restarts repeatedly during model loading.
3. The HPA shows unknown CPU utilization.
4. The image builds but `/health` returns 503.

After testing, remove the local container and explain how you would remove the
Kubernetes resources without accidentally deleting shared cluster resources.

In [ ]:
# TODO: Provide commands and explanations for all four symptoms.
# TODO: Local cleanup example: docker stop sentiment-api
# TODO: Kubernetes cleanup example: kubectl delete -f k8s/hpa.yaml -f k8s/service.yaml -f k8s/deployment.yaml ...

## Submission Checklist

- API contract and lifecycle explanation completed.
- Local or Docker endpoint tests documented.
- Dockerfile and Kubernetes manifests analyzed.
- Deployment, probes, storage, and HPA explained.
- Troubleshooting answers and cleanup commands included.

# Solutions

Use this section only after attempting all TODO cells.

## Solution 1 — API contract and lifecycle

`MODEL_NAME` is read from the environment and defaults to
`distilbert-base-uncased-finetuned-sst-2-english`. The lifespan handler loads
the Transformers sentiment pipeline once when the application starts. `/health`
returns 503 until loading completes; `/predict` accepts a non-empty list of
texts and returns one label and score per text. Loading at startup avoids
reloading the model for every request.

In [ ]:
print((APP_DIR / "app/main.py").read_text())

## Solution 2 — Local and Docker testing

```bash
uvicorn app.main:app --host 0.0.0.0 --port 8000
curl http://localhost:8000/health
curl -X POST http://localhost:8000/predict \
  -H 'Content-Type: application/json' \
  -d '{"texts":["I love this service", "This is terrible"]}'

docker build -t sentiment-api:week9 .
docker run --rm --name sentiment-api -p 8000:8000 sentiment-api:week9
docker images sentiment-api:week9
docker logs sentiment-api
```

## Solution 3 — Manifest summary and deployment analysis

The Namespace isolates the application resources. The ConfigMap supplies
`MODEL_NAME` and `HF_HOME` without rebuilding the image. The Deployment runs
two replicas of `quay.io/ml-app/sentiment-api:latest`, exposes port 8000,
mounts the model-cache PVC, and requests 2 CPU/2 GiB while limiting each pod
to 4 CPU/4 GiB. Startup allows model loading time; readiness controls Service
traffic; liveness restarts an unhealthy container.

In [ ]:
for path in sorted((APP_DIR / "k8s").glob("*.yaml")):
    document = yaml.safe_load(path.read_text())
    print(path.name, document.get("kind"), document.get("metadata", {}).get("name"),
          document.get("metadata", {}).get("namespace", "default"))

## Solution 4 — Kubernetes deployment

```bash
kubectl apply -f k8s/namespace.yaml
kubectl apply -f k8s/configmap.yaml -f k8s/pv.yaml \
  -f k8s/deployment.yaml -f k8s/service.yaml -f k8s/hpa.yaml
kubectl -n ml-app get pods,svc,deploy,hpa,pvc
kubectl -n ml-app rollout status deployment/ml-app
kubectl -n ml-app logs deployment/ml-app
kubectl -n ml-app port-forward service/ml-app 8000:80
```

The PV provides storage, the PVC requests and binds that storage, and the
Deployment mounts it at `/app/models`. `hostPath` is node-local, so it is not a
portable production storage solution for pods scheduled on different nodes.

## Solution 5 — HPA and troubleshooting

The HPA keeps between 2 and 8 replicas and targets average CPU utilization of
70% for the `ml-app` Deployment. CPU alone may not represent model latency,
queue depth, or request rate, so production scaling may need custom metrics.

- Not in Service endpoints: run `kubectl -n ml-app describe pod POD` and check selector labels and readiness probe events.
- Repeated restarts: run `kubectl -n ml-app logs POD --previous`; inspect memory limits and model-loading errors.
- Unknown HPA metric: run `kubectl top pods -n ml-app`; verify Metrics Server is installed.
- Health returns 503: inspect container logs; the model is still loading or failed to load.

Clean up only the named application resources:

```bash
kubectl delete -f k8s/hpa.yaml -f k8s/service.yaml -f k8s/deployment.yaml \
  -f k8s/configmap.yaml -f k8s/pv.yaml
kubectl delete namespace ml-app
```